# Smart IDS v3 — Notebook 07: Automated Mitigation Engine & Active Response Simulation
Threat Tiers, IP Validation, Idempotent Dry-Run Blocking, and SOC Audit Logging

---

### What You Will Learn
1. **The Detection-to-Response Gap**: Why alerts without automated mitigation fail against automated attacks.
2. **Four-Tier Threat Severity Taxonomy**: Mapping confidence scores to risk-proportionate mitigation actions:
   - Low (0.00 – 0.40): LOG
   - Medium (0.40 – 0.70): ALERT
   - High (0.70 – 0.90): RATE_LIMIT
   - Critical (0.90 – 1.01): BLOCK
3. **Defensive Programming in Cybersecurity**: Validating IP addresses with `ipaddress` to prevent command injection.
4. **Firewall Rule Construction**: Understanding Linux `iptables` DROP rules (`-I INPUT 1 -s <IP> -j DROP`).
5. **Idempotency & Simulation**: Ensuring dry-run safety and preventing duplicate firewall degradation.
6. **Append-Only Audit Logging**: Tracking security responses in `ids_response_log.txt`.

---

### Prerequisites
- Completion of Notebooks 01 through 06.
- Understanding of the Single Canonical Module design (`src/smart_ids/response.py`).


## 1. Security Concepts: Automated Mitigation & Command Injection Defense

> **Security Concept: What is iptables and why DROP instead of REJECT?**
> `iptables` is the Linux kernel's packet filtering firewall. The command:
> `iptables -I INPUT 1 -s 198.51.100.2 -j DROP`
> - `-I INPUT 1`: Inserts this rule at the very top (position 1) of the inbound packet chain, guaranteeing immediate precedence.
> - `-j DROP`: Silently discards any packet from this source IP with **zero reply**.
> 
> In contrast, `-j REJECT` sends an ICMP "port unreachable" error packet back to the sender. This tells an attacker that an active firewall is blocking them. `DROP` leaves them hanging, wasting their connection time and leaking zero information.

> **Security Concept: Defensive IP Validation against Command Injection**
> If an attacker submits a malicious flow with an IP like:
> `192.168.1.1; rm -rf /`
> and we interpolate that into a system shell command, the server executes arbitrary malware!
> 
> *The Defense:* We pass every candidate IP through Python's standard `ipaddress.ip_address()` validator. If it fails to parse as a valid IPv4 or IPv6 address, the block is aborted immediately with action code `BLOCK_FAILED`.


In [1]:
import sys
from pathlib import Path

cur = Path.cwd()
root = cur if (cur / 'src').exists() else cur.parent
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

from smart_ids.config import (
    LOGS_FILE,
    THREAT_ACTIONS,
    THREAT_COLORS,
    THREAT_THRESHOLDS,
)
from smart_ids.response import (
    apply_active_response,
    get_threat_level,
    reset_blocked_ips,
    validate_ip,
)

# Reset blocked IP cache for this demonstration session
reset_blocked_ips()
print(f"Active Response Engine initialized.")
print(f"Audit log destination: {LOGS_FILE}")


Active Response Engine initialized.
Audit log destination: D:\SMART_IDS\ids_response_log.txt


## 2. Testing the 4-Tier Severity Mapping

**What we do and why:**
We verify that confidence scores correctly map to the 4 severity tiers:
- $0.25 	o$ Low
- $0.55 	o$ Medium
- $0.80 	o$ High
- $0.95 	o$ Critical
- Exactly $1.00 	o$ Critical (ensured by the interval $[0.90, 1.01)$).


In [2]:
test_confidences = [0.15, 0.40, 0.65, 0.70, 0.85, 0.90, 0.99, 1.00]

print("--- Confidence to Severity Mapping ---")
for conf in test_confidences:
    tier = get_threat_level(conf)
    action = THREAT_ACTIONS[tier]
    print(f"Confidence: {conf:4.2f} -> Tier: {tier:8s} -> Action: {action}")


--- Confidence to Severity Mapping ---
Confidence: 0.15 -> Tier: Low      -> Action: LOG
Confidence: 0.40 -> Tier: Medium   -> Action: ALERT
Confidence: 0.65 -> Tier: Medium   -> Action: ALERT
Confidence: 0.70 -> Tier: High     -> Action: RATE_LIMIT
Confidence: 0.85 -> Tier: High     -> Action: RATE_LIMIT
Confidence: 0.90 -> Tier: Critical -> Action: BLOCK
Confidence: 0.99 -> Tier: Critical -> Action: BLOCK
Confidence: 1.00 -> Tier: Critical -> Action: BLOCK


## 3. Testing Defensive IP Address Validation

**What we do and why:**
We test `validate_ip` against valid IPv4 and IPv6 addresses, as well as malformed injection payloads.


In [3]:
test_ips = [
    "192.168.1.100",
    "10.0.0.1",
    "2001:0db8:85a3:0000:0000:8a2e:0370:7334",
    "INVALID_IP",
    "192.168.1.1; cat /etc/passwd",
    "999.999.999.999",
    "",
]

print("--- IP Address Validation Checks ---")
for candidate in test_ips:
    valid = validate_ip(candidate)
    status = "VALID" if valid else "REJECTED (SAFE)"
    print(f"[{status:15s}] '{candidate}'")


--- IP Address Validation Checks ---
[VALID          ] '192.168.1.100'
[VALID          ] '10.0.0.1'
[VALID          ] '2001:0db8:85a3:0000:0000:8a2e:0370:7334'
[REJECTED (SAFE)] 'INVALID_IP'
[REJECTED (SAFE)] '192.168.1.1; cat /etc/passwd'
[REJECTED (SAFE)] '999.999.999.999'
[REJECTED (SAFE)] ''


## 4. Simulating Graduated Incident Scenarios

Now let's simulate realistic detection scenarios across all 4 threat tiers.


In [4]:
scenarios = [
    {"ip": "10.0.0.15", "class": "BENIGN", "conf": 0.22, "model": "SmartIDS_MLP"},
    {"ip": "198.51.100.42", "class": "PortScan", "conf": 0.58, "model": "SmartIDS_1DCNN"},
    {"ip": "203.0.113.88", "class": "SSH-Patator", "conf": 0.82, "model": "XGBoost"},
    {"ip": "192.0.2.1", "class": "DoS Hulk", "conf": 0.96, "model": "SmartIDS_MLP"},
    {"ip": "INVALID_INJECTION_IP", "class": "DDoS", "conf": 0.98, "model": "SmartIDS_1DCNN"},
]

print("--- Simulated Incident Response Execution ---")
for s in scenarios:
    res = apply_active_response(
        src_ip=s["ip"],
        attack_class=s["class"],
        confidence=s["conf"],
        model_name=s["model"],
        dry_run=True,
    )
    print(f"\nIncident: [{res['threat_level']}] from {res['src_ip']}")
    print(f"  Class: {res['attack_class']} (Confidence: {res['confidence']:.2f})")
    print(f"  Decision: {res['action']}")
    print(f"  Detail:   {res['action_detail']}")


--- Simulated Incident Response Execution ---

Incident: [Low] from 10.0.0.15
  Class: BENIGN (Confidence: 0.22)
  Decision: LOG
  Detail:   Flow logged for baseline statistical analysis. Confidence 0.22 < 0.40.

Incident: [Medium] from 198.51.100.42
  Class: PortScan (Confidence: 0.58)
  Decision: ALERT
  Detail:   Security event dispatched to SIEM and SOC dashboard. Class: PortScan.

Incident: [High] from 203.0.113.88
  Class: SSH-Patator (Confidence: 0.82)
  Decision: RATE_LIMIT
  Detail:   Rate limiting rule enforced: traffic from 203.0.113.88 restricted to 100 pps.

Incident: [Critical] from 192.0.2.1
  Class: DoS Hulk (Confidence: 0.96)
  Decision: BLOCK
  Detail:   [DRY RUN] Simulated iptables command: iptables -I INPUT 1 -s 192.0.2.1 -j DROP

Incident: [Critical] from INVALID_INJECTION_IP
  Class: DDoS (Confidence: 0.98)
  Decision: BLOCK_FAILED
  Detail:   IP validation failed for candidate 'INVALID_INJECTION_IP'. Blocking aborted to prevent injection.


## 5. Idempotency Gate Verification

> **Security Concept: What is Idempotency in Firewall Management?**
> If an attacker bombards a server with 50,000 requests in 2 seconds, the IDS will detect 50,000 malicious flows from the same IP.
> If the engine adds a new `iptables` rule 50,000 times, the firewall's memory and packet traversal performance will collapse!
> 
> *The Solution:* The engine maintains an in-memory set `BLOCKED_IPS`. If an IP is already blocked, subsequent calls immediately return without issuing redundant rules.


In [5]:
attacker_ip = "192.0.2.1"

# Second call for the same IP
repeat_res = apply_active_response(
    src_ip=attacker_ip,
    attack_class="DoS Hulk",
    confidence=0.99,
    dry_run=True,
)

print(f"Repeat Block Attempt for {attacker_ip}:")
print(f"  Action: {repeat_res['action']}")
print(f"  Detail: {repeat_res['action_detail']}")


Repeat Block Attempt for 192.0.2.1:
  Action: BLOCK
  Detail: IP 192.0.2.1 is already blocked. Skipping redundant rule (idempotent).


## 6. Inspecting the Audit Trail (`ids_response_log.txt`)

Let's read the last 10 entries from our durable, append-only security log file.


In [6]:
if LOGS_FILE.exists():
    with open(LOGS_FILE, 'r', encoding='utf-8') as f:
        log_lines = f.readlines()
    print(f"--- Last {min(8, len(log_lines))} Log Entries from {LOGS_FILE.name} ---")
    for line in log_lines[-8:]:
        print(line.strip())
else:
    print("Log file will be created upon first event.")


--- Last 8 Log Entries from ids_response_log.txt ---
[2026-11-17 01:14:25] [CRITICAL] ACTION=BLOCK [SIMULATED] | IP=198.51.100.77 | RULE='iptables -I INPUT 1 -s 198.51.100.77 -j DROP'
[2026-11-17 01:14:25] [INFO] IP 198.51.100.77 is already blocked. Skipping redundant rule (idempotent).
[2026-11-17 01:59:42] [INFO] ACTION=LOG | IP=10.0.0.15 | Class=BENIGN | Conf=0.2200
[2026-11-17 01:59:42] [INFO] ACTION=ALERT | IP=198.51.100.42 | Class=PortScan | Conf=0.5800
[2026-11-17 01:59:42] [WARNING] ACTION=RATE_LIMIT | IP=203.0.113.88 | Limit=100 pps
[2026-11-17 01:59:42] [CRITICAL] ACTION=BLOCK [SIMULATED] | IP=192.0.2.1 | RULE='iptables -I INPUT 1 -s 192.0.2.1 -j DROP'
[2026-11-17 01:59:42] [ERROR] ACTION=BLOCK_FAILED | Invalid IP='INVALID_INJECTION_IP'
[2026-11-17 01:59:42] [INFO] IP 192.0.2.1 is already blocked. Skipping redundant rule (idempotent).


---
## Summary & Research Series Conclusion

Congratulations! Across the 7 notebooks in the **Smart IDS v3** curriculum, you have:
1. Explored network flows and the CICIDS2017 benchmark dataset.
2. Cleaned zero-division infinities and built a leak-free scaling + SMOTE pipeline.
3. Trained three supervised architectures: XGBoost, Deep MLP, and 1D-CNN.
4. Built an unsupervised Autoencoder for zero-day anomaly detection.
5. Benchmarked microsecond latency, normalized confusion matrices, and ROC curves.
6. Decoded model predictions using SHAP game-theoretic feature attribution.
7. Deployed an active response engine with injection-safe, idempotent firewall simulation.

All trained models and metadata are saved in `models/` and ready for the interactive **Streamlit SOC Dashboard** (`app/streamlit_app.py`)!
